# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Lane 4, CTR opportunity scoring. The rule in plain words:** a page is worth a title/meta review first if it's **visible** (ranks on page 1 or 2 with real impressions) AND its **CTR sits far below the median CTR of its position tier**. Rank those pages by the clicks they would have earned at the tier median.

Both halves are the signals behind FlyRank's own flags from the session:

- **CTR vs position** is the signal behind the CTR-fix logic. If CTR tracks position very tightly, there's nothing to review; if it varies a lot inside a tier, the gap is a real place to look.
- **Volume (impressions)** is the signal behind the quick-win flag. Below a floor, one click swings CTR too much.

Each signal gets a bucket table with `n` and a one-word verdict, so I know the rule rests on something before I encode it.

### Signal 1: CTR vs position (behind CTR-fix)

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/nadim7009/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import json, pathlib, pandas as pd, numpy as np
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages from {df['client_id'].nunique()} clients")

30,000 pages from 32 clients


In [2]:
# Pages with real position data and enough impressions for CTR to mean something
has_pos = df["avg_position"] > 0
visible = df[has_pos & (df["impressions_90d"] >= 500)].copy()
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

ctr_by_tier = (visible.groupby("position_tier", observed=True)
                      .agg(n=("ctr", "size"),
                           p25=("ctr", lambda s: s.quantile(0.25)),
                           median_ctr=("ctr", "median"),
                           p75=("ctr", lambda s: s.quantile(0.75)))
                      .reindex(order))
print("CTR (%) by position tier, impressions_90d >= 500:")
print(ctr_by_tier.round(2))

p1 = ctr_by_tier.loc["page_1"]
print(f"\npage_1: top-quarter CTR / bottom-quarter CTR = {p1['p75'] / max(p1['p25'], 1e-6):.1f}x")
print("Verdict (CTR-vs-position signal): MIXED")
print(" - Median CTR does drop deeper into the results (page_1 0.24% -> deep 0.00%), as expected.")
print(" - But inside page_1, CTR at the 75th percentile is 3.7x the 25th. So position alone leaves a lot unexplained,")
print("   and the within-tier gap is a real thing to score on.")
print(" - A caveat: the top_3 tier shows median 0.20% vs page_1's 0.24%, which is backwards (small tier, 458 pages).")

CTR (%) by position tier, impressions_90d >= 500:
                  n   p25  median_ctr   p75
position_tier                              
top_3           458  0.08        0.20  0.49
page_1         7064  0.12        0.24  0.44
striking       4485  0.08        0.17  0.34
page_3_5       4330  0.01        0.09  0.19
deep            389  0.00        0.00  0.04

page_1: top-quarter CTR / bottom-quarter CTR = 3.7x
Verdict (CTR-vs-position signal): MIXED
 - Median CTR does drop deeper into the results (page_1 0.24% -> deep 0.00%), as expected.
 - But inside page_1, CTR at the 75th percentile is 3.7x the 25th. So position alone leaves a lot unexplained,
   and the within-tier gap is a real thing to score on.
 - A caveat: the top_3 tier shows median 0.20% vs page_1's 0.24%, which is backwards (small tier, 458 pages).


### Signal 2: staleness (behind the refresh flag)

Does `days_since_last_update` actually line up with `trend_direction == "down"`? The refresh flag assumes it does.

In [3]:
df["declining"] = df["trend_direction"].eq("down").astype(int)
bins = [-1, 30, 90, 180, 365, 100000]
labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["stale_tier"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

stale_vs_down = (df.groupby("stale_tier", observed=True)
                   .agg(n=("declining", "size"), share_declining=("declining", "mean"))
                   .round(3))
print("Share of pages trending down, by days-since-last-update tier:")
print(stale_vs_down)

print("\nVerdict (staleness signal): FALSE")
print(" - The share of declining pages barely moves across staleness tiers (0.51, 0.59, 0.61, 0.47, 0.60).")
print(" - On this slice, 'older pages decline more' isn't supported. The 181-365 tier even dips to 0.47.")
print(" - That's a clearly-explained negative, and it saved my rule: I'm NOT putting staleness in it.")
print(" - The rule leans only on CTR-vs-position (signal 1).")

Share of pages trending down, by days-since-last-update tier:
                n  share_declining
stale_tier                        
0-30        20480            0.511
31-90         175            0.589
91-180       9171            0.611
181-365       169            0.467
365+            5            0.600

Verdict (staleness signal): FALSE
 - The share of declining pages barely moves across staleness tiers (0.51, 0.59, 0.61, 0.47, 0.60).
 - On this slice, 'older pages decline more' isn't supported. The 181-365 tier even dips to 0.47.
 - That's a clearly-explained negative, and it saved my rule: I'm NOT putting staleness in it.
 - The rule leans only on CTR-vs-position (signal 1).


## 2. Build the ranked queue (writes the CSV)

**The rule, encoded.** Candidate pool: pages with `avg_position` in (0, 20] and `impressions_90d >= 500` (the volume floor from the session).

```text
expected_ctr      = median CTR of the page's position_tier (fitted on the pool)
click_gap_score   = max(expected_ctr - ctr, 0) / 100 * impressions_90d
reason_code       = "ctr_below_tier_median"  (only when ctr < 0.5 * expected_ctr)
action_label      = "review_title_meta"      (only when the reason code fires)
```

A page below half its tier's median CTR gets the reason code and the action label. Everything else stays scored but unflagged, so a reviewer can see where the next-tier candidates sit. The CSV holds the top 200.

In [4]:
pool = df[has_pos & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].copy()
pool["expected_ctr"] = pool.groupby("position_tier", observed=True)["ctr"].transform("median")
pool["click_gap_score"] = ((pool["expected_ctr"] - pool["ctr"]).clip(lower=0) / 100 * pool["impressions_90d"]).round(1)

below_half = pool["ctr"] < 0.5 * pool["expected_ctr"]
pool["reason_code"] = np.where(below_half, "ctr_below_tier_median", "")
pool["action_label"] = np.where(below_half, "review_title_meta", "monitor")

queue = (pool.sort_values("click_gap_score", ascending=False)
             [["content_id", "client_id", "position_tier", "avg_position", "impressions_90d",
               "clicks_90d", "ctr", "expected_ctr", "click_gap_score", "reason_code", "action_label"]])

OUT = pathlib.Path("work/outputs"); OUT.mkdir(parents=True, exist_ok=True)
CSV = OUT / "baseline_action_score.csv"
queue.head(200).to_csv(CSV, index=False)
print(f"Wrote {CSV}  ({len(queue):,} pages scored, top 200 saved)")
print(f"Flagged for review (reason + action):  {(pool['action_label'] == 'review_title_meta').sum():,} pages")

# Metrics receipts committed to git (the CSV itself is gitignored by design)
receipts = {
    "pool_pages": int(len(pool)),
    "clients": int(pool["client_id"].nunique()),
    "flagged_review": int((pool["action_label"] == "review_title_meta").sum()),
    "top50_flag_rate": float((queue.head(50)["action_label"] == "review_title_meta").mean()),
    "top50_zero_clicks": int((queue.head(50)["clicks_90d"] == 0).sum()),
    "ctr_vs_position_verdict": "MIXED",
    "staleness_verdict": "FALSE",
    "rule": "visible AND ctr < 0.5 * tier_median_ctr; ranked by click_gap_score"
}
json.dump(receipts, open(OUT / "baseline_metrics.json", "w"), indent=2)
print("Wrote work/outputs/baseline_metrics.json")

Wrote work/outputs/baseline_action_score.csv  (12,023 pages scored, top 200 saved)
Flagged for review (reason + action):  3,031 pages
Wrote work/outputs/baseline_metrics.json


## 3. Top-10 review

For each row: the action, why it's there (the reason code and the gap), and one line on what would make it wrong.

In [5]:
top10 = queue.head(10).reset_index(drop=True)
view = top10[["content_id", "client_id", "position_tier", "avg_position", "impressions_90d", "clicks_90d",
              "ctr", "expected_ctr", "click_gap_score", "action_label"]]
print(view.round(2).to_string())

print("\nPer-row review (what would make this call wrong):")
tier_median_source = pool.groupby("position_tier", observed=True)["ctr"].median().to_dict()
for i, r in top10.iterrows():
    tier_med = tier_median_source[r["position_tier"]]
    reason = "no reason code (gap below the 50% cut)" if not r["reason_code"] else r["reason_code"]
    wrong = ("CTR dropped because of a SERP feature (AI summary, image pack) that took the click, "
             "not because the title is bad") if r["ctr"] < tier_med else "position may have moved mid-window, so the tier label is wrong"
    if r["clicks_90d"] == 0:
        wrong = "page may be de-indexed or redirecting; zero clicks across 90d can mean 'no result shown' not 'poor title'"
    if r["impressions_90d"] > 50000:
        wrong = "the page may rank for mostly branded queries, where title changes carry much more risk than reward"
    print(f"{i+1:>2}. [{r['action_label']}] pos_tier={r['position_tier']} impr={int(r['impressions_90d']):>6,} "
          f"ctr={r['ctr']:.2f}% vs tier median {tier_med:.2f}% -> gap_score={r['click_gap_score']:.0f}  | "
          f"reason: {reason}  | wrong-if: {wrong}")

             content_id          client_id position_tier  avg_position  impressions_90d  clicks_90d   ctr  expected_ctr  click_gap_score       action_label
0  content_36ff89c8214e  client_19581e27de        page_1           7.3           295097         154  0.05          0.24            560.7  review_title_meta
1  content_5fe46e04994d  client_4e07408562        page_1           4.2           517715         741  0.14          0.24            517.7            monitor
2  content_c8e9d6ab9013  client_19581e27de        page_1           9.7           208678           0  0.00          0.24            500.8  review_title_meta
3  content_c84a0ab98e90  client_f369cb89fc        page_1           7.8           223271          70  0.03          0.24            468.9  review_title_meta
4  content_8451fc6f034d  client_d029fa3a95         top_3           2.3           272144          75  0.03          0.20            462.6  review_title_meta
5  content_453722754fea  client_f369cb89fc        page_1        

## 4. Weak picks + leakage check

**Leakage check.** The rule uses only `avg_position`, `position_tier`, `impressions_90d` and `ctr`, all measured in the same 90-day window as what I'm trying to flag now. No `trend_direction`, no `trend_pct`, and no `is_declining_label` touch it. The flag is a comparison inside the current window, not a prediction of a future outcome. That means the baseline is honest for "which page looks worth a review right now", not for "which page will later turn around if touched". The modeling week will add a future-window label on the warehouse.

**Weak picks the top-10 made visible:**

- The whole top of the queue sits in `page_1`, because that's where the volume is. A reviewer could easily spend all their time on one tier.
- The `top_3` tier median is lower than `page_1`'s on this slice, so any `top_3` page in the queue is scored against an unreliable expectation. I'll either hold `top_3` out or compute its expected CTR from the warehouse later.
- Pages with 50,000+ impressions are often branded queries; a bad title there costs more than it saves. The review-wrong line names that risk per row.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.